#### Telematics Data Generation

#### NOC Real-Time Fleet & Tanker Monitoring Dashboard – ECTS-Style

**Strategic Context**  
This notebook generates realistic GPS telematics data simulating NOC's Electronic Cargo Tracking System (ECTS).  
Focus: Fuel tankers departing from the strategic Dukem Fuels Depot to key delivery locations across Ethiopia.

**Industry Standards Applied**:
- High-frequency GPS pings (every 5 minutes)
- Real tanker parameters (fuel level, valve status, odometer, harsh events)
- Reproducible and modular design

#### Path Setup & Imports

In [1]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from geopy.distance import geodesic
import os
import warnings

warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

print("Libraries imported and project path configured.")

Libraries imported and project path configured.


#### Load Configuration

In [2]:
from config import VEHICLES, ROUTES, NUM_DAYS, SAMPLES_PER_HOUR, START_DATE

print(f"NOC Configuration Loaded")
print(f"   • {len(VEHICLES)} vehicles (own + subcontracted)")
print(f"   • {len(ROUTES)} strategic routes from Dukem Depot")

NOC Configuration Loaded
   • 6 vehicles (own + subcontracted)
   • 6 strategic routes from Dukem Depot


#### Data Generation Function (Industry-Standard Telematics)

In [3]:
def generate_telematics_data():
    data = []
    start_time = pd.to_datetime(START_DATE)
    
    for vehicle in VEHICLES:
        for day in range(NUM_DAYS):
            current_time = start_time + timedelta(days=day)
            route_name = random.choice(list(ROUTES.keys()))
            route = ROUTES[route_name]
            
            total_points = int(route["typical_duration_hours"] * SAMPLES_PER_HOUR * 1.25)
            
            lat = route["start"][0] + random.uniform(-0.012, 0.012)
            lon = route["start"][1] + random.uniform(-0.012, 0.012)
            
            current_fuel = random.uniform(84.0, 97.5) if vehicle["capacity_liters"] > 0 else None
            odometer = random.uniform(45200, 178500)
            engine_hours = random.uniform(1720, 4980)
            trip_id = f"{vehicle['vehicle_id']}_{current_time.strftime('%Y%m%d')}"
            
            is_moving = True
            last_lat, last_lon = lat, lon
            
            for i in range(total_points):
                progress = min(i / total_points * random.uniform(0.96, 1.04), 1.0)
                target_lat, target_lon = route["end"]
                
                current_lat = lat + (target_lat - lat) * progress + random.uniform(-0.011, 0.011)
                current_lon = lon + (target_lon - lon) * progress + random.uniform(-0.011, 0.011)
                
                dist_since_last = geodesic((last_lat, last_lon), (current_lat, current_lon)).km
                
                speed = random.uniform(40, 82) if is_moving else random.uniform(0, 7)
                
                if is_moving and random.random() < 0.16 and i > 10:
                    is_moving = False
                elif not is_moving and random.random() < 0.40:
                    is_moving = True
                
                if current_fuel is not None:
                    consumption = 0.68 if is_moving else 0.26
                    current_fuel = max(7.0, current_fuel - consumption * random.uniform(0.9, 1.25))
                
                valve_status = "CLOSED"
                if not is_moving and random.random() < 0.028:
                    valve_status = "OPEN"
                
                ignition_status = "ON" if speed > 6 or random.random() < 0.78 else "OFF"
                
                harsh_braking = 1 if is_moving and random.random() < 0.037 else 0
                harsh_acceleration = 1 if is_moving and random.random() < 0.031 else 0
                
                device_status = "ONLINE" if random.random() < 0.98 else "OFFLINE"
                
                data.append({
                    "timestamp": current_time.replace(microsecond=0),
                    "vehicle_id": vehicle["vehicle_id"],
                    "vehicle_type": vehicle["type"],
                    "owner": vehicle["owner"],
                    "trip_id": trip_id,
                    "latitude": round(current_lat, 6),
                    "longitude": round(current_lon, 6),
                    "speed_kmh": round(speed, 1),
                    "odometer_km": round(odometer + dist_since_last, 1),
                    "fuel_level_percent": round(current_fuel, 2) if current_fuel is not None else None,
                    "valve_status": valve_status,
                    "ignition_status": ignition_status,
                    "engine_hours": round(engine_hours + 0.0083, 2),
                    "route": route_name,
                    "is_moving": is_moving,
                    "harsh_braking": harsh_braking,
                    "harsh_acceleration": harsh_acceleration,
                    "device_status": device_status
                })
                
                last_lat, last_lon = current_lat, current_lon
                current_time += timedelta(minutes=5)
                odometer += dist_since_last
                engine_hours += 0.0083
    
    return pd.DataFrame(data)


print("Generating high-quality NOC ECTS telematics data...")
df_raw = generate_telematics_data()
print(f"Generated {len(df_raw):,} records")

Generating high-quality NOC ECTS telematics data...
Generated 4,260 records


#### Save Raw Data & Basic Inspection

In [4]:
os.makedirs('../data/raw', exist_ok=True)
df_raw.to_csv('../data/raw/noc_telematics_raw.csv', index=False)

print("Raw data saved to ../data/raw/noc_telematics_raw.csv")

display(df_raw.head())
print("\nRecords per vehicle:\n", df_raw['vehicle_id'].value_counts())

Raw data saved to ../data/raw/noc_telematics_raw.csv


,timestamp,vehicle_id,vehicle_type,owner,trip_id,latitude,longitude,speed_kmh,odometer_km,fuel_level_percent,valve_status,ignition_status,engine_hours,route,is_moving,harsh_braking,harsh_acceleration,device_status
0,2025-04-01 06:00:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.800370,38.922388,41.3,63800.7,86.64,CLOSED,ON,2054.14,Dukem_to_Adama,True,0,0,ONLINE
1,2025-04-01 06:05:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.791891,38.927145,51.7,63801.7,85.85,CLOSED,ON,2054.15,Dukem_to_Adama,True,0,0,ONLINE
2,2025-04-01 06:10:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.794186,38.926773,56.0,63802.0,85.15,CLOSED,ON,2054.16,Dukem_to_Adama,True,0,0,ONLINE
3,2025-04-01 06:15:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.792310,38.933654,52.3,63802.8,84.33,CLOSED,ON,2054.17,Dukem_to_Adama,True,0,0,ONLINE
4,2025-04-01 06:20:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.782612,38.962432,75.9,63806.1,83.63,CLOSED,ON,2054.18,Dukem_to_Adama,True,0,0,ONLINE



Records per vehicle:
 vehicle_id
NOC-V501    870
NOC-T001    825
NOC-T002    825
SUB-T102    675
NOC-T003    540
SUB-T101    525
Name: count, dtype: int64
